# Update time series with linear updater

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
from sweat.timeseries import status_handler as sh
from sweat.timeseries.linear_updater import LinearUpdater, LinearUpdaterParams
from sweat.timeseries.types import TimeSeriesVar as TSVar

In [ ]:
config = {"parallel": True, "num_workers": 8}

In [ ]:
LinearUpdaterParams.model_validate(config)

### Initiate data

Create the dataset `data`

In [ ]:
# Generate 7 dates until now
window = 7
today = pd.Timestamp.now()
dates = pd.date_range(end=today, periods=window, freq="D").date

# Create dimensions
size_x = 2
size_y = 3
x = np.arange(size_x)
y = np.arange(size_y)

shape = (size_y, size_x, window)

# Create data variables
et_data = np.full(shape, np.nan)
radiation_data = np.ones(shape)
updated_data = np.zeros(shape, dtype=bool)
state_data = np.full(shape, sh.State.NODATA.value, dtype=sh.STATE_TYPE)
distance_data = np.full(shape, sh.DISTANCE_MAX, dtype=sh.DISTANCE_TYPE)
flags_data = np.zeros(shape, dtype=sh.VALIDITY_FLAGS_TYPE)

# Create the dataset
data = xr.Dataset(
    data_vars={
        TSVar.ET.value: (["y", "x", "time"], et_data),
        TSVar.RADIATION.value: (["y", "x", "time"], radiation_data),
        TSVar.UPDATED.value: (["y", "x", "time"], updated_data),
        TSVar.STATE.value: (["y", "x", "time"], state_data),
        TSVar.DISTANCE.value: (["y", "x", "time"], distance_data),
        TSVar.VALIDITY_FLAGS.value: (["y", "x", "time"], flags_data),
    },
    coords={
        "time": dates,
        "x": x,
        "y": y,
    },
)

In [ ]:
# Specify tests
data[TSVar.ET.value].loc[{"x": 0, "y": 2}] = np.array(
    [2, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]
)
data[TSVar.STATE.value].loc[{"x": 0, "y": 2}] = np.array([2, 7, 7, 7, 7, 7, 7])
data[TSVar.DISTANCE.value].loc[{"x": 0, "y": 2}] = np.array(
    [2, 255, 255, 255, 255, 255, 255], dtype=np.uint8
)
data[TSVar.ET.value].loc[{"x": 1, "y": 0}] = np.array(
    [5, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]
)
data[TSVar.STATE.value].loc[{"x": 1, "y": 0}] = np.array(
    [2, 7, 7, 7, 7, 7, 7], dtype=np.int8
)
data[TSVar.DISTANCE.value].loc[{"x": 1, "y": 0}] = np.array(
    [6, 255, 255, 255, 255, 255, 255], dtype=np.uint8
)
data[TSVar.ET.value].loc[{"x": 1, "y": 1}] = np.array(
    [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 11]
)
data[TSVar.STATE.value].loc[{"x": 1, "y": 1}] = np.array([7, 7, 7, 7, 7, 7, 3])
data[TSVar.DISTANCE.value].loc[{"x": 1, "y": 1}] = np.array(
    [255, 255, 255, 255, 255, 255, 12], dtype=np.uint8
)
data[TSVar.ET.value].loc[{"x": 1, "y": 2}] = np.array(
    [2, np.nan, np.nan, np.nan, np.nan, np.nan, 8]
)
data[TSVar.STATE.value].loc[{"x": 1, "y": 2}] = np.array([2, 7, 7, 7, 7, 7, 3])
data[TSVar.DISTANCE.value].loc[{"x": 1, "y": 2}] = np.array(
    [2, 255, 255, 255, 255, 255, 2], dtype=np.uint8
)

Create the second dataset `feed` which corresponds to new acquisitions to take into account

In [ ]:
# Create ET time series
def et_piecewise_function(x):
    return np.where(x < 0, x + 6, -0.5 * x + 6)


# Calculate the corresponding y values using the piecewise function
et = et_piecewise_function(np.linspace(-3, 3, window))

In [ ]:
acquisition_dates = [3, 6]
feed_dates = dates[acquisition_dates]
feed_et = et[acquisition_dates]

In [ ]:
# Create an xarray dataset
feed = xr.Dataset(
    {
        TSVar.ET.value: (
            ["y", "x", "time"],
            np.tile(feed_et[np.newaxis, np.newaxis, :], (size_y, size_x, 1)),
        ),
        TSVar.VALID.value: (
            ["y", "x", "time"],
            np.ones((size_y, size_x, len(feed_dates))),
        ),
    },
    coords={
        "time": feed_dates,
        "x": x,
        "y": y,
    },
)

In [ ]:
# Set specific cases
feed[TSVar.VALID.value].loc[{"x": 0, "y": 0}] = np.array([1, 1])
feed[TSVar.VALID.value].loc[{"x": 0, "y": 1}] = np.array([1, 0])
feed[TSVar.VALID.value].loc[{"x": 0, "y": 2}] = np.array([0, 1])
feed[TSVar.VALID.value].loc[{"x": 1, "y": 0}] = np.array([0, 0])
feed[TSVar.VALID.value].loc[{"x": 1, "y": 1}] = np.array([0, 0])
feed[TSVar.VALID.value].loc[{"x": 1, "y": 2}] = np.array([0, 0])

### LinearUpdater

In [ ]:
linear_updater = LinearUpdater()

In [ ]:
updated_data = data.copy()

Update new data

In [ ]:
new_et, new_state, new_updated, new_distance = (
    linear_updater.update_with_new_acquisitions(data, feed)
)

In [ ]:
for y in np.arange(size_y):
    print(f"For pixel x=0, y={y}")
    print(f"  - ET: {new_et.isel(x=0, y=y).values}")
    print(f"  - State: {new_state.isel(x=0, y=y).values}")
    print(f"  - Updated: {new_updated.isel(x=0, y=y).values}\n")

In [ ]:
updated_data[TSVar.ET.value] = new_et
updated_data[TSVar.STATE.value] = new_state
updated_data[TSVar.DISTANCE.value] = new_distance
updated_data[TSVar.UPDATED] = new_updated

Find next and find previous

In [ ]:
state = updated_data[TSVar.STATE.value].isel(x=slice(0, 1), y=slice(0, 1))
print(f"State: {state.isel(x=0, y=0).values}")
print(
    f"Next: {
        linear_updater.find_next(state, (state == sh.State.ACQUISITION.value))
        .isel(x=0, y=0)
        .values
    }"
)

In [ ]:
state = updated_data[TSVar.STATE.value].isel(x=slice(0, 1), y=slice(0, 1))
print(f"State: {state.isel(x=0, y=0).values}")
print(
    f"Previous: {
        linear_updater.find_previous(
            state, (state == sh.State.ACQUISITION.value)
        )
        .isel(x=0, y=0)
        .values
    }"
)

In [ ]:
save_data = updated_data.copy()

In [ ]:
updated_data.daily_radiation.notnull()

Update time series (2 steps)

In [ ]:
new_et, new_updated, new_state, new_distance = (
    linear_updater.update_with_acquisitions(
        et=updated_data[TSVar.ET.value],
        radiation=updated_data[TSVar.RADIATION.value],
        state=updated_data[TSVar.STATE.value],
        distance=updated_data[TSVar.DISTANCE.value],
    )
)

In [ ]:
for x in np.arange(size_x):
    for y in np.arange(size_y):
        print(f"For pixel x={x}, y={y}")
        print(
            f"  - Original state: {updated_data[TSVar.STATE.value].isel(x=x, y=y).values}"
        )
        print(f"  - ET: {new_et.isel(x=x, y=y).values}")
        print(f"  - State: {new_state.isel(x=x, y=y).values}")
        print(f"  - Distance: {new_distance.isel(x=x, y=y).values}")
        print(f"  - Updated: {new_updated.isel(x=x, y=y).values}\n")

In [ ]:
updated_data[TSVar.ET.value] = new_et
updated_data[TSVar.STATE.value] = new_state
updated_data[TSVar.DISTANCE.value] = new_distance
updated_data[TSVar.UPDATED.value] |= new_updated  # Update existing one

In [ ]:
new_et, new_updated, new_state, new_distance = (
    linear_updater.update_with_extrapolation_only(
        et=updated_data[TSVar.ET.value],
        radiation=updated_data[TSVar.RADIATION.value],
        state=updated_data[TSVar.STATE.value],
        distance=updated_data[TSVar.DISTANCE.value],
    )
)

In [ ]:
for x in np.arange(size_x):
    for y in np.arange(size_y):
        print(f"For pixel x={x}, y={y}")
        print(
            f"  - Original state: {updated_data[TSVar.STATE.value].isel(x=x, y=y).values}"
        )
        print(f"  - ET: {new_et.isel(x=x, y=y).values}")
        print(f"  - State: {new_state.isel(x=x, y=y).values}")
        print(f"  - Distance: {new_distance.isel(x=x, y=y).values}")
        print(f"  - Updated: {new_updated.isel(x=x, y=y).values}\n")

In [ ]:
updated_data[TSVar.ET.value] = new_et
updated_data[TSVar.STATE.value] = new_state
updated_data[TSVar.DISTANCE.value] = new_distance
updated_data[TSVar.UPDATED.value] |= new_updated  # Update existing one

In [ ]:
for x in np.arange(size_x):
    for y in np.arange(size_y):
        print(f"For pixel x={x}, y={y}")
        print(f"  - ET: {updated_data[TSVar.ET.value].isel(x=x, y=y).values}")
        print(
            f"  - State: {updated_data[TSVar.STATE.value].isel(x=x, y=y).values}"
        )
        print(
            f"  - Distance: {updated_data[TSVar.DISTANCE.value].isel(x=x, y=y).values}"
        )
        print(
            f"  - Updated: {updated_data[TSVar.UPDATED.value].isel(x=x, y=y).values}\n"
        )

Update time series

In [ ]:
updated_data = linear_updater.update_time_series(save_data)

In [ ]:
for x in np.arange(size_x):
    for y in np.arange(size_y):
        print(f"For pixel x={x}, y={y}")
        print(f"  - ET: {updated_data[TSVar.ET.value].isel(x=x, y=y).values}")
        print(
            f"  - State: {updated_data[TSVar.STATE.value].isel(x=x, y=y).values}"
        )
        print(
            f"  - Distance: {updated_data[TSVar.DISTANCE.value].isel(x=x, y=y).values}"
        )
        print(
            f"  - Updated: {updated_data[TSVar.UPDATED.value].isel(x=x, y=y).values}\n"
        )

Update the dataset

In [ ]:
updated_data = linear_updater.update(data, feed)

In [ ]:
for x in np.arange(size_x):
    for y in np.arange(size_y):
        print(f"For pixel x={x}, y={y}")
        print(f"  - ET: {updated_data[TSVar.ET.value].isel(x=x, y=y).values}")
        print(
            f"  - State: {updated_data[TSVar.STATE.value].isel(x=x, y=y).values}"
        )
        print(
            f"  - Distance: {updated_data[TSVar.DISTANCE.value].isel(x=x, y=y).values}"
        )
        print(
            f"  - Updated: {updated_data[TSVar.UPDATED.value].isel(x=x, y=y).values}\n"
        )